# S14 — Modelos recurrentes y de atención ligeros (punto 4 de la revisión del docente)
Compara, **con las mismas particiones, datos sin artefacto (protocolo C) y las mismas 25 entradas de IM** de MI-LiteXGB, dos líneas base profundas adicionales:

- **LSTM compacta** (32 unidades; cada atributo es un paso de una secuencia de 25), ≈ 4,8 mil parámetros.
- **Transformer compacto** (tokens por atributo con token [CLS], 2 bloques, 2 cabezas, d = 16, FFN 32), ≈ 5,6 mil parámetros.

Ambos del tamaño de la TCN de Gao (4 123 parámetros). Se entrenan con Adam (10⁻³), entropía cruzada, lotes de 512, hasta 50 épocas con
**parada temprana** (pérdida de validación, paciencia 5, mejores pesos) y 10 % de validación, sin balanceo, como la TCN.

| Parte | Qué hace | Dónde | Tiempo aprox. |
|---|---|---|---|
| **E8** | Entrena y evalúa LSTM y Transformer en cada REP (y exporta a ONNX los de la REP-01) | **GPU** con RAM alta | ≈ 8–15 min por REP y modelo |
| **E9** | Latencia por paquete (lote de 1 y de 1 024, ONNX Runtime, 1 hilo) de LSTM, Transformer, TCN, MI-LiteRF y MI-LiteXGB en la misma sesión | CPU (sirve la misma sesión) | ≈ 5 min |
| **E10** | Resumen y pruebas pareadas (t remuestreada corregida + Holm) frente a MI-LiteXGB y a la TCN con las mismas entradas | Cualquiera | < 1 min |

**Cómo correrlo:** sesión **GPU con RAM alta** (L4 o A100) → *Ejecutar todo*. Si no alcanza el tiempo, `REPS = [1, 2, 3, 4, 5]` ya da una comparación
útil (y luego `[6, 7, 8, 9, 10]` en otra sesión). Con `REANUDAR = True` retoma donde quedó. Las secciones 0 a 3.1 son copia **sin cambios** de S11e.
Este cuaderno **no cambia** resultados anteriores: lee `corridas/` y `mejoras/` y escribe todo en `mejoras/S14_recurrentes/`.
Al final, **comparte `S14_recurrentes_resultados.json`** (o la salida de E10).

## 0. Configuración

In [ ]:
import os
# --- Qué se ejecuta -------------------------------------------------------------------------------
EJECUTAR = {'E8': True, 'E9': True, 'E10': True}   # E8 necesita GPU; E9 y E10, CPU
REPS = list(range(1, 11))   # si falta tiempo: [1, 2, 3, 4, 5] en una sesión y [6, 7, 8, 9, 10] en otra
REANUDAR = True            # reutiliza lo ya guardado en Drive (por REP y por parte)

# --- Rutas ----------------------------------------------------------------------------------------
CARPETA = os.environ.get('CARPETA_ARTICULO', '/content/drive/MyDrive/Articulo_Grupal_EdgeIIoT')
RUTA_CSV = os.environ.get('EDGE_CSV', os.path.join(CARPETA, 'DNN-EdgeIIoT-dataset.csv'))
DIR_V1 = os.path.join(CARPETA, 'corridas')                 # resultados v1 (solo lectura)
DIR_MEJ = os.path.join(CARPETA, 'mejoras')                 # todo lo nuevo
DIR_TMP = os.environ.get('DIR_TMP_S14R', '/content/tmp_s14r')   # disco local: datos de medición y modelos de paso
NB_CONSOLIDACION = os.environ.get('NB_CONSOLIDACION', '/content/drive/MyDrive/Colab Notebooks/S11_Consolidacion_Estadistica.ipynb')

# --- Parámetros de v1 (sección 13; no modificar) ---------------------------------------------------
TEST_SIZE = 0.30
SMOTE_K = 5
MI_SUBMUESTRA = 200_000
MI_MAX_DISCRETO = 64
MI_BINS = 32
LOTE_MEDICION = 1024
V1_VERSIONES = {'scikit-learn': '1.6.1', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1'}   # las de las fichas v1
COLS_RETIRAR = ['frame.time', 'ip.dst_host', 'http.file_data', 'ip.src_host', 'arp.src.proto_ipv4',
                'arp.dst.proto_ipv4', 'http.request.full_uri', 'http.request.uri.query',
                'icmp.transmit_timestamp', 'tcp.payload', 'tcp.options', 'tcp.srcport',
                'tcp.dstport', 'udp.port', 'mqtt.msg']
COLS_CATEGORICAS = ['http.request.method', 'http.referer', 'http.request.version',
                    'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic']
GAO_18 = ['icmp.checksum', 'icmp.seq_le', 'http.response', 'tcp.ack', 'tcp.ack_raw',
          'tcp.connection.rst', 'tcp.connection.syn', 'udp.stream', 'dns.qry.name',
          'http.request.method-0', 'http.request.method-GET', 'http.request.method-POST',
          'http.referer-0', 'http.request.version-0', 'dns.qry.name.len-0.0',
          'mqtt.conack.flags-0.0', 'mqtt.protoname-0.0', 'mqtt.topic-0.0']
NOMBRE = {'tcn': 'TCN', 'rf': 'MI-LiteRF', 'xgb': 'MI-LiteXGB'}
ALFA = 0.05

# --- M1: benchmark de despliegue ---------------------------------------------------------------------
M1_N_LOTE1 = 10_000          # flujos para la latencia con lote de 1 en ONNX Runtime (mediana por flujo)
M1_N_LOTE1_NATIVO = 2_000    # en el runtime de v1 (Keras / scikit-learn / XGBoost), solo como referencia
M1_CALENTAMIENTO_LOTE1 = 500 # llamadas de calentamiento descartadas antes de medir con lote de 1
M1_OPSET = 17                # opset ONNX para la TCN y el RF (onnxmltools admite hasta 15 para XGBoost)

# --- M2: ablación (regla de lectura fijada antes) ------------------------------------------------------
MARGEN_RELEVANTE = 1.0       # puntos de F1 macro: diferencia mínima con relevancia práctica
#   Por brazo: Δ = F1(MI-LiteRF) − F1(brazo), prueba pareada bilateral (Shapiro → t o Wilcoxon), Holm sobre los 3 brazos.
#   «A favor de MI-LiteRF» si p Holm < 0,05 y Δ ≥ margen; «en contra» si p Holm < 0,05 y Δ ≤ −margen; si no, «sin diferencia relevante».

# --- M3: MI-LiteRF v2 compacto (REGLA PENDIENTE DE APROBACIÓN) ---------------------------------------
M3_K_GRID = [10, 15, 18, 20, 25]
M3_PROF_GRID = [8, 10, 12, 15]
M3_ARBOLES_GRID = [10, 15, 25]
M3_T = 2000                  # balanceo de v1
M3_TOLERANCIA = 0.99         # candidatos: F1 macro CV ≥ 99 % del mejor de la rejilla
M3_K_MAX = 18                # ... y no más atributos que la TCN
M3_FILTRO_SU = True          # True: ranking IM + descarte de redundantes por incertidumbre simétrica (SU)
M3_TAU_SU = 0.99             # un atributo es redundante si SU ≥ 0,99 con uno ya elegido
M3_CV_POR_REP = True         # True: CV dentro de cada REP (como v1). False: CV solo en REP-01 y se aplica a todas
M3_CRITERIO_TAMANO = 'tamano_kb'   # 'tamano_kb' = joblib sin comprimir (indicador 14.3); 'tamano_c3_kb' = joblib compress=3
REGLA_M3_APROBADA = True     # aprobada por Leonardo el 16/11/2026: opción B (IM + filtro SU) y CV en cada REP
M3_VISTA_PREVIA = True       # muestra el ranking con filtro SU de la 1.ª REP (solo entrenamiento) aunque M3 esté bloqueada

# --- M4: sensibilidad al artefacto «0»/«0.0» (opcional) ------------------------------------------------
M4_CONFIG = 'v1'             # 'v1' (k, T, RF y XGB de v1) o 'v2' (la configuración elegida en M3)

# --- S2: MI-LiteRF v3, balanceo elegido por CV (regla fijada antes de mirar el conjunto de prueba) ----------
S2_T_GRID = [2000, 5000, 10000, 20000, None]   # None = sin balanceo (entrenamiento completo)
S2_TOLERANCIA = 0.99                           # misma parsimonia de v1: F1 CV ≥ 99 % del mejor; el más pequeño (joblib)
REGLA_S2_APROBADA = False                      # 🔒

# --- S3/S4/S5: escenario sin artefacto (regla fijada antes de mirar el conjunto de prueba) -------------------
# TCN de v1 (S10, sección 7), sin cambios: mismos hiperparámetros y mismos 18 atributos de Gao, con las variables «-0.0»
# reemplazadas por su versión unificada «-0» (al unificar «0.0» → «0», la variable «-0.0» deja de existir).
TCN_CFG = dict(lr=1e-3, epocas=50, lote=128, dilataciones=[1, 2, 4, 8], kernel=1, filtros=26, dropout=0.1, val_split=0.1)
S3_MAPEO_GAO = {'dns.qry.name.len-0.0': 'dns.qry.name.len-0', 'mqtt.conack.flags-0.0': 'mqtt.conack.flags-0',
                'mqtt.protoname-0.0': 'mqtt.protoname-0', 'mqtt.topic-0.0': 'mqtt.topic-0'}
S5_TFP_MAX = 1.0                               # % — índice «óptimo» de la matriz 10.2
S5_TAU_GRID = [round(0.05 * i, 2) for i in range(1, 20)]   # 0,05 … 0,95 (además de «argmax»)
S3_EXIGIR_GPU = True                           # la TCN necesita GPU (en CPU tardaría horas por REP)
REGLA_S3_APROBADA = False                      # 🔒

# --- S11e: análisis para el paper (FGCS) ---------------------------------------------------------------
E4_CONJUNTOS = ['gao18', 'sin_id', 'sin_id_cont', 'todos']   # ablaciones de atributos (árboles de v3, sin balanceo)
ID_COLS = ['tcp.ack_raw', 'tcp.checksum', 'icmp.checksum', 'udp.stream']   # identificadores propios de cada captura
CONT_COLS = ['tcp.seq', 'tcp.ack', 'icmp.seq_le']                           # contadores de sesión
E4_K = 25                    # mismo k de v1/v3
E4_MEDIR_LATENCIA = True     # latencia por paquete (lote de 1, ONNX Runtime, 1 hilo) de cada ablación
E5_VARIANTES = ['tcn_cw', 'tcn_mi25']   # TCN con pesos por clase + parada temprana; TCN con los 25 atributos de IM
E5_PACIENCIA = 5             # épocas sin mejorar la pérdida de validación antes de parar (solo tcn_cw)
E2_UMBRAL_SEPARADOR = 0.999  # un atributo es «separador» si por sí solo acierta Normal/Ataque en ≥ 99,9 % de los paquetes

# --- S14: modelos recurrentes y de atención (punto 4 de la revisión) -------------------------------------
E8_VARIANTES = ['lstm', 'transformer']
E8_CFG = dict(lr=1e-3, epocas=50, lote=512, val_split=0.1, paciencia=5)       # Adam, parada temprana (mejores pesos)
E8_LSTM = dict(unidades=32)                                                   # ≈ 4,8 mil parámetros
E8_TRANSFORMER = dict(d=16, cabezas=2, capas=2, ffn=32, dropout=0.1)          # ≈ 5,6 mil parámetros
E9_N_LOTE1 = 10_000          # paquetes para la latencia con lote de 1 (mediana)
E9_CALENTAMIENTO = 500

os.makedirs(DIR_TMP, exist_ok=True)      # disco local; las carpetas de Drive se crean después de montarlo (0.1)
print('REPS:', REPS, '| partes:', [p for p, v in EJECUTAR.items() if v], '| salida:', DIR_MEJ)

## 0.1 Drive, versiones y librerías
Se fijan scikit-learn, imbalanced-learn y XGBoost en las versiones de las fichas v1: así los `.joblib` guardados cargan sin cambios y el balanceo se reproduce exactamente. Si Colab trae otras versiones, la celda las instala y te pide reiniciar la sesión una vez.

In [ ]:
import sys, subprocess, json, time, platform, datetime, io, gc, glob, gzip, shutil, warnings, math
import importlib.metadata as im
from types import SimpleNamespace
EN_COLAB = 'google.colab' in sys.modules
ficha_rep1 = os.path.join(DIR_V1, 'REP-01', 'ficha_REP-01.json')
if EN_COLAB and not os.path.exists(ficha_rep1):          # Drive aún no montado (si ya se montó desde el panel, no se vuelve a pedir)
    # Solo se quitan carpetas VACÍAS que este notebook pudo crear antes de montar (os.rmdir nunca borra archivos)
    for d in (DIR_MEJ, CARPETA, '/content/drive/MyDrive', '/content/drive'):
        try:
            os.rmdir(d)
        except OSError:
            pass
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        raise SystemExit(f'⛔ No se pudo montar Drive ({e}). Móntalo desde el panel Archivos (ícono de carpeta → '
                         'botón de Drive) y vuelve a ejecutar esta celda.')
if not os.path.exists(ficha_rep1):
    raise SystemExit(f'⛔ No encuentro {ficha_rep1}. Revisa que el Drive montado sea el de la cuenta dueña de '
                     'Articulo_Grupal_EdgeIIoT (Entorno de ejecución → Desconectar y borrar entorno, y vuelve a montar).')
os.makedirs(DIR_MEJ, exist_ok=True)

def version(paquete):
    try:
        return im.version(paquete)
    except im.PackageNotFoundError:
        return None

distintas = {p: version(p) for p, v in V1_VERSIONES.items() if version(p) != v}
if distintas:
    ya_cargadas = [m for m in ('sklearn', 'imblearn', 'xgboost') if m in sys.modules]
    print('Instalando las versiones de v1 (ahora hay', distintas, ')')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *[f'{p}=={v}' for p, v in V1_VERSIONES.items()]], check=True)
    if ya_cargadas:
        raise SystemExit('⛔ Se cambiaron librerías que ya estaban cargadas. Reinicia la sesión '
                         '(Entorno de ejecución → Reiniciar sesión) y vuelve a ejecutar desde el inicio.')
faltan = [p for p in ('onnx', 'onnxruntime', 'skl2onnx', 'onnxmltools', 'papermill', 'kagglehub', 'openpyxl') if version(p) is None]
if faltan:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan,
                    *[f'{p}=={v}' for p, v in V1_VERSIONES.items()]], check=True)
if version('tf2onnx') is None:   # sin dependencias para no tocar protobuf ni TensorFlow
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'tf2onnx'], check=True)

import numpy as np, pandas as pd, psutil, joblib
import sklearn, imblearn, xgboost, scipy
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

def cpu_modelo():
    try:
        for l in open('/proc/cpuinfo'):
            if l.startswith('model name'):
                return l.split(':', 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

ENTORNO = {
    'fecha_hora': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'),
    'evaluadores': 'Leonardo Leoncio Villanueva Fonseca; Ricardo Franco Baldoceda Rios',
    'archivo_datos': os.path.basename(RUTA_CSV),
    'procesador': cpu_modelo(), 'nucleos_logicos': psutil.cpu_count(),
    'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 2),
    'medicion': 'CPU, 1 hilo, un proceso por modelo',
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__, 'imbalanced_learn': imblearn.__version__,
    'xgboost': xgboost.__version__, 'tensorflow': version('tensorflow'), 'keras': version('keras'),
    'scipy': scipy.__version__, 'onnx': version('onnx'), 'onnxruntime': version('onnxruntime'),
    'skl2onnx': version('skl2onnx'), 'onnxmltools': version('onnxmltools'), 'tf2onnx': version('tf2onnx'),
}

# Comparación con el entorno de las fichas v1
ENTORNO_V1 = json.load(open(ficha_rep1, encoding='utf-8'))['tabla_14_2']
claves = ['procesador', 'nucleos_logicos', 'ram_total_gb', 'python', 'numpy', 'pandas', 'scikit_learn',
          'imbalanced_learn', 'xgboost', 'tensorflow', 'keras', 'scipy']
comp = pd.DataFrame({'v1 (REP-01)': [ENTORNO_V1.get(c) for c in claves], 'esta sesión': [ENTORNO.get(c) for c in claves]}, index=claves)
comp['igual'] = ['✔' if str(a) == str(b) else '≠' for a, b in zip(comp.iloc[:, 0], comp.iloc[:, 1])]
for c in ('onnx', 'onnxruntime', 'skl2onnx', 'onnxmltools', 'tf2onnx'):
    comp.loc[c] = ['—', ENTORNO[c], '']
if shutil.which('nvidia-smi') and 'GPU' in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout:
    print('ℹ️ Hay GPU en la sesión: solo la usa E5 (TCN). E1–E4, E6 y E7 corren en CPU.')
comp

## 0.2 Utilidades comunes
Funciones copiadas de S10 sin cambios (entropía, discretización, balanceo, indicadores) y las pruebas pareadas del notebook de consolidación.

In [ ]:
def guardar_json(obj, ruta):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2,
                  default=lambda o: o.item() if hasattr(o, 'item') else (o.tolist() if hasattr(o, 'tolist') else str(o)))

def leer_json(ruta):
    with open(ruta, encoding='utf-8') as f:
        return json.load(f)

def hay(ruta):
    return REANUDAR and os.path.exists(ruta)

def correr(parte):
    if not EJECUTAR.get(parte):
        print(f'⏭️ {parte} desactivada en EJECUTAR.')
        return False
    if parte == 'M3' and not REGLA_M3_APROBADA:
        print('🔒 M3 bloqueada: falta aprobar la regla (REGLA_M3_APROBADA = True en la configuración).')
        return False
    return True

def bitacora(parte, rep, inicio, nota=''):
    ruta = os.path.join(DIR_MEJ, f'bitacora_{parte}.csv')
    fila = pd.DataFrame([{'parte': parte, 'rep': rep, 'inicio': time.strftime('%d/%m/%Y %H:%M:%S', time.localtime(inicio)),
                          'segundos': round(time.time() - inicio, 1), 'nota': nota}])
    fila.to_csv(ruta, mode='a', header=not os.path.exists(ruta), index=False)

def kb_joblib(modelo, compress=0):
    b = io.BytesIO(); joblib.dump(modelo, b, compress=compress); return b.tell() / 1024

def kb_gzip(ruta):
    with open(ruta, 'rb') as f:
        return len(gzip.compress(f.read(), compresslevel=9)) / 1024

# --- Información mutua (S10, sección 8) ---------------------------------------------------------------
def entropia(*codigos):
    c = codigos[0].astype(np.int64)
    for otro in codigos[1:]:                       # entropía conjunta: código combinado
        c = c * (int(otro.max()) + 1) + otro.astype(np.int64)
    _, cnt = np.unique(c, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

def discretizar(x):
    u = np.unique(x)
    if len(u) <= MI_MAX_DISCRETO:
        return np.searchsorted(u, x)
    bordes = np.unique(np.quantile(x, np.linspace(0, 1, MI_BINS + 1)[1:-1]))
    return np.digitize(x, bordes)

# --- Balanceo (S10, sección 9): submuestreo → SMOTE → Tomek -------------------------------------------
def balancear(Xs, ys, T, semilla):
    cnt = np.bincount(ys, minlength=N_CLASES)
    bajo = {c: T for c in range(N_CLASES) if cnt[c] > T}
    if bajo:
        Xs, ys = RandomUnderSampler(sampling_strategy=bajo, random_state=semilla).fit_resample(Xs, ys)
    cnt = np.bincount(ys, minlength=N_CLASES)
    sobre = {c: T for c in range(N_CLASES) if 0 < cnt[c] < T}
    if sobre:
        kn = min(SMOTE_K, int(cnt[list(sobre)].min()) - 1)
        Xs, ys = SMOTE(sampling_strategy=sobre, k_neighbors=kn, random_state=semilla).fit_resample(Xs, ys)
    Xs, ys = TomekLinks(sampling_strategy='all', n_jobs=-1).fit_resample(Xs, ys)
    return np.ascontiguousarray(Xs, dtype=np.float32), ys

# --- Indicadores (S10, sección 13; el retardo sigue «no calculable», como en v1) ------------------------
def indicadores(y_real, y_pred):
    cm = confusion_matrix(y_real, y_pred, labels=range(N_CLASES))
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0) * 100
    TN = cm[NORMAL, NORMAL]; FP = cm[NORMAL].sum() - TN
    return {'f1_macro': float(f1.mean()), 'f1_por_clase': dict(zip(CLASES, f1.tolist())),
            'tfp': float(FP / (FP + TN) * 100), 'falsas_alarmas': int(FP),
            'exactitud': float(accuracy_score(y_real, y_pred) * 100),
            'f1_ponderado': float(f1_score(y_real, y_pred, average='weighted') * 100),
            'matriz_confusion': cm.tolist(),
            'retardo_mediana_ms': 'no calculable', 'episodios_no_detectados': 'no calculable'}

def f1_macro(y_real, y_pred):
    return f1_score(y_real, y_pred, average='macro', labels=range(N_CLASES), zero_division=0) * 100

# --- Pruebas pareadas + Holm (iguales al notebook de consolidación) -------------------------------------
def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    orden = np.argsort(p, kind='mergesort'); ajust = np.empty(m); acumulado = 0.0
    for i, idx in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (m - i) * p[idx])); ajust[idx] = acumulado
    return ajust

def prueba_pareada(o1, o2, alternativa, alfa=ALFA):
    """Shapiro–Wilk de d = O2 − O1; t pareada si es normal, si no Wilcoxon. alternativa: 'two-sided', 'greater', 'less'."""
    o1, o2 = np.asarray(o1, float), np.asarray(o2, float)
    r = {'n': int(len(o1)), 'media_O1': float(np.mean(o1)), 'media_O2': float(np.mean(o2)), 'alternativa': alternativa,
         'media_d': float(np.mean(o2 - o1)), 'de_d': float(np.std(o2 - o1, ddof=1)) if len(o1) > 1 else np.nan,
         'shapiro_p': np.nan, 'prueba': '—', 'estadistico': np.nan, 'gl': np.nan, 'p': 1.0, 'dz': np.nan,
         'ic95_d': (np.nan, np.nan), 'evaluable': False}
    d = o2 - o1
    if len(d) < 3 or np.allclose(d, 0):
        r['estado'] = 'Sin diferencias' if len(d) >= 3 else 'n < 3'
        return r
    r['evaluable'] = True
    normal = False
    if r['de_d'] > 0:
        r['shapiro_p'] = float(stats.shapiro(d).pvalue); normal = r['shapiro_p'] > alfa
    if r['de_d'] > 0:
        h = stats.t.ppf(0.975, len(d) - 1) * r['de_d'] / np.sqrt(len(d))
        r['ic95_d'] = (r['media_d'] - h, r['media_d'] + h)
    if normal:
        t = stats.ttest_rel(o2, o1, alternative=alternativa)
        r.update(prueba='t pareada', estadistico=float(t.statistic), gl=int(len(d) - 1), p=float(t.pvalue),
                 dz=r['media_d'] / r['de_d'])
    else:
        w = stats.wilcoxon(d, alternative=alternativa)
        r.update(prueba='Wilcoxon', estadistico=float(w.statistic), p=float(w.pvalue))
    r['estado'] = 'Evaluada'
    return r

def media_de(s):
    s = pd.Series(s, dtype=float)
    return f'{s.mean():.4g} ± {s.std(ddof=1):.2g}' if len(s) > 1 else f'{s.mean():.4g}'

# --- Estilo de figuras (el mismo de la sección 16) --------------------------------------------------------
COLOR = {'tcn': '#2a78d6', 'rf': '#eb6834', 'xgb': '#1baf7a'}
MARCA = {'tcn': 'o', 'rf': 's', 'xgb': '^'}
TINTA, TINTA2, REJILLA = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.edgecolor': TINTA2, 'axes.labelcolor': TINTA,
                     'xtick.color': TINTA2, 'ytick.color': TINTA2, 'axes.titlesize': 10})
coma = matplotlib.ticker.FuncFormatter(lambda x, _: f'{x:g}'.replace('.', ','))

## 1. Preprocesamiento común (idéntico a S10, secciones 3 y 4)
Se lee el CSV, se retiran los 15 atributos, faltantes y duplicados, y se codifican los 7 categóricos con los nombres de Gao. `frame.time` no se lee: en v1 se apartaba antes de quitar faltantes y duplicados, así que el resultado es el mismo. Con `unificar_ceros=True` (solo M4) se unifica «0.0» → «0» **después** de quitar duplicados, para conservar exactamente las mismas filas y la misma división de v1.

In [ ]:
def preprocesar(unificar_ceros=False):
    t0 = time.perf_counter()
    if not os.path.exists(RUTA_CSV):
        import kagglehub
        ruta = kagglehub.dataset_download('mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot',
                                          path='Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv')
        globals()['RUTA_CSV'] = ruta if ruta.endswith('.csv') else glob.glob(os.path.join(ruta, '**', 'DNN-EdgeIIoT-dataset.csv'), recursive=True)[0]
    retirar = set(COLS_RETIRAR)
    df = pd.read_csv(RUTA_CSV, low_memory=False, usecols=lambda c: c.strip() not in retirar)
    df.columns = df.columns.str.strip()
    n_bruto, n_col_tras_retiro = len(df), df.shape[1] - 2      # sin Attack_label ni Attack_type
    df = df.dropna(axis=0, how='any')
    n_tras_faltantes = len(df)
    df = df.drop_duplicates(keep='first')
    n_depurado = len(df)
    for c in COLS_CATEGORICAS:
        df[c] = df[c].astype(str).str.strip()
        if unificar_ceros:
            df[c] = df[c].replace({'0.0': '0'})
    dummies = pd.get_dummies(df[COLS_CATEGORICAS], prefix=COLS_CATEGORICAS, prefix_sep='-', dtype=np.float32)
    numericas = df.drop(columns=COLS_CATEGORICAS + ['Attack_label', 'Attack_type'])
    for c in [c for c in numericas.columns if not pd.api.types.is_numeric_dtype(numericas[c])]:
        numericas[c] = pd.to_numeric(numericas[c], errors='coerce')
    X_df = pd.concat([numericas.astype(np.float32), dummies], axis=1)
    y_txt = df['Attack_type'].astype(str).str.strip().to_numpy()
    del df, numericas, dummies
    finitas = np.isfinite(X_df.to_numpy()).all(axis=1)
    if (~finitas).any():
        X_df, y_txt = X_df[finitas], y_txt[finitas]
    le = LabelEncoder().fit(y_txt)
    atributos = X_df.columns.tolist()
    B = SimpleNamespace(X=X_df.to_numpy(np.float32), y=le.transform(y_txt).astype(np.int64), ATRIBUTOS=atributos,
                        CLASES=le.classes_.tolist(), unificar_ceros=unificar_ceros)
    B.preproc = {'registros_brutos': n_bruto, 'tras_faltantes': n_tras_faltantes, 'registros_depurados': int(len(B.y)),
                 'duplicados_eliminados': n_tras_faltantes - n_depurado, 'filas_no_finitas': int((~finitas).sum()),
                 'atributos_tras_retiro': n_col_tras_retiro, 'dimensiones_tras_codificacion': len(atributos), 'clases': B.CLASES, 'unificar_ceros': unificar_ceros,
                 'esperado_gao': {'registros_depurados': 1909671, 'dimensiones': 97}}
    print(f'{"Unificado «0»/«0.0»" if unificar_ceros else "Codificación v1"}: {len(B.y):,} flujos | '
          f'{len(atributos)} dimensiones | {time.perf_counter() - t0:.0f} s')
    return B

BASE = preprocesar(False)
X_CRUDO, Y = BASE.X, BASE.y
ATRIBUTOS, CLASES = BASE.ATRIBUTOS, BASE.CLASES
N_CLASES, NORMAL = len(CLASES), CLASES.index('Normal')
IDX_GAO = [ATRIBUTOS.index(c) for c in GAO_18]

# Control: lo mismo que registró v1
pp1 = json.load(open(ficha_rep1, encoding='utf-8'))['preprocesamiento']
assert len(Y) == pp1['registros_depurados'], f'Flujos depurados: {len(Y):,} ≠ v1 {pp1["registros_depurados"]:,}'
assert len(ATRIBUTOS) == pp1['dimensiones_tras_codificacion'], 'Dimensiones distintas de v1'
assert CLASES == pp1['clases'], 'Clases distintas de v1'
print('✔ Igual a v1:', f'{len(Y):,} flujos, {len(ATRIBUTOS)} dimensiones, {N_CLASES} clases')

## 2. División por repetición, artefactos v1 y verificación
`dividir(r)` reproduce la división 70/30 estratificada con semilla 1000 + r y el min-max ajustado solo con el entrenamiento. `verificar_v1` comprueba con el `rf.joblib` guardado que el F1 macro sale **idéntico** al de `indicadores.json`: si no, se detiene, porque significaría que el conjunto de prueba no es el mismo.

In [ ]:
def dividir(r, B=None):
    B = B or BASE
    semilla = 1000 + r
    idx_tr, idx_te = train_test_split(np.arange(len(B.y)), test_size=TEST_SIZE, stratify=B.y, random_state=semilla)
    esc = MinMaxScaler().fit(B.X[idx_tr])
    return SimpleNamespace(r=r, codigo=f'REP-{r:02d}', semilla=semilla, idx_tr=idx_tr, idx_te=idx_te, esc=esc,
                           Xtr=esc.transform(B.X[idx_tr]).astype(np.float32), Xte=esc.transform(B.X[idx_te]).astype(np.float32),
                           ytr=B.y[idx_tr], yte=B.y[idx_te])

def v1(r):
    d = os.path.join(DIR_V1, f'REP-{r:02d}')
    ficha = leer_json(os.path.join(d, f'ficha_REP-{r:02d}.json'))
    sel = ficha['seleccion']
    mi = pd.read_csv(os.path.join(d, 'mi_ranking.csv'))
    orden = mi['indice'].to_numpy()
    K = int(sel['k'])
    assert [ATRIBUTOS[j] for j in orden[:K]] == sel['atributos_k'], f'REP-{r:02d}: el ranking no coincide con la ficha'
    return SimpleNamespace(dir=d, ficha=ficha, ind=leer_json(os.path.join(d, 'indicadores.json')), mi=mi, orden_mi=orden,
                           K=K, cols_k=orden[:K], T=int(sel['T']), rf=(int(sel['rf_arboles']), int(sel['rf_profundidad'])),
                           xgb=(int(sel['xgb_arboles']), int(sel['xgb_profundidad'])), xgb_lr=float(sel.get('xgb_lr', 0.1)),
                           ruta_tcn=os.path.join(d, 'tcn.keras'), ruta_rf=os.path.join(d, 'rf.joblib'),
                           ruta_xgb=os.path.join(d, 'xgb.joblib'))


def verificar_v1(D, V):
    prueba_v1 = {x['Clase']: x['Flujos de prueba'] for x in V.ficha['tabla_14_4'] if not str(x['Clase']).startswith('Total')}
    assert dict(zip(CLASES, np.bincount(D.yte, minlength=N_CLASES).tolist())) == prueba_v1, f'{D.codigo}: flujos de prueba ≠ v1'
    rf = joblib.load(V.ruta_rf); rf.set_params(n_jobs=1)   # 1 hilo, como la medición de v1: suma los árboles en el mismo orden
    f1 = f1_macro(D.yte, rf.predict(D.Xte[:, V.cols_k]))
    dif = abs(f1 - V.ind['rf']['f1_macro'])
    assert dif < 0.01, f'{D.codigo}: F1 de MI-LiteRF reconstruido = {f1:.6f} ≠ v1 {V.ind["rf"]["f1_macro"]:.6f}'
    if dif > 1e-6:
        print(f'⚠️ {D.codigo}: F1 reconstruido difiere de v1 en {dif:.6f} puntos (empates numéricos); se continúa.')
    return f1

def guardar_x(ruta, M):
    np.save(ruta, np.ascontiguousarray(M, dtype=np.float32)); return ruta

def lote1_npz(ruta, D, sub, cols):
    """Flujos para la latencia con lote de 1: escalados (inferencia) y crudos (preprocesamiento + inferencia)."""
    cols = np.asarray(cols)
    np.savez(ruta, x_escalado=D.Xte[sub][:, cols], x_crudo=X_CRUDO[D.idx_te[sub]], cols=cols, indices=sub,
             min_=D.esc.min_[cols], scale_=D.esc.scale_[cols])
    return ruta

D_PRUEBA = dividir(REPS[0]); V_PRUEBA = v1(REPS[0])
print(f'✔ {D_PRUEBA.codigo}: entrenamiento {len(D_PRUEBA.ytr):,} | prueba {len(D_PRUEBA.yte):,} | '
      f'F1 MI-LiteRF reconstruido = {verificar_v1(D_PRUEBA, V_PRUEBA):.4f} (idéntico a v1)')
del D_PRUEBA; gc.collect()

## 2.1 Utilidades de S11d (candados, balanceo por CV, umbral τ, cota del retardo)

In [ ]:
# S11d: candados, utilidades y escenario sin artefacto
def correr(parte):
    if not EJECUTAR.get(parte):
        print(f'⏭️ {parte} desactivada en EJECUTAR.')
        return False
    candado = {'S2': REGLA_S2_APROBADA, 'S3': REGLA_S3_APROBADA}.get(parte, True)
    if not candado:
        print(f'🔒 {parte} bloqueada: lee su regla y pon REGLA_{parte}_APROBADA = True en la configuración.')
        return False
    return True

DIR_M2 = os.path.join(DIR_MEJ, 'M2_ablacion')
DIR_S2 = os.path.join(DIR_MEJ, 'S2_balanceo_v3')
DIR_S3 = os.path.join(DIR_MEJ, 'S3_sin_artefacto')
ATAQUES = [c for c in range(N_CLASES) if c != NORMAL]
TODAS = set(range(1, 11))

def bitacora(parte, rep, inicio, nota=''):
    """Como en S11b, pero un archivo por parte y REP: varias sesiones en paralelo no escriben el mismo archivo de Drive."""
    ruta = os.path.join(DIR_MEJ, 'bitacoras_S11d', f'{parte}_REP-{rep:02d}.csv')
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    pd.DataFrame([{'parte': parte, 'rep': rep, 'inicio': time.strftime('%d/%m/%Y %H:%M:%S', time.localtime(inicio)),
                   'segundos': round(time.time() - inicio, 1), 'nota': nota}]).to_csv(ruta, index=False)

def nivel_R(x):
    return 'Crítico' if x < 25 else ('Aceptable' if x <= 75 else 'Óptimo')

def T_txt(T):
    return 'sin balanceo' if not T else f'{int(T):,}'.replace(',', ' ')

def datos_T(Xs, ys, T, semilla):
    """T = None o 0: sin balanceo (entrenamiento completo, distribución natural)."""
    if not T:
        return np.ascontiguousarray(Xs, dtype=np.float32), ys
    return balancear(Xs, ys, int(T), semilla)

def cota_retardo(cm):
    """Cota del retardo de M7: por clase de ataque, M_c = flujos de c clasificados como Normal."""
    cm = np.asarray(cm)
    filas = []
    for c in ATAQUES:
        n_c, M_c = int(cm[c].sum()), int(cm[c, NORMAL])
        m_c = M_c / n_c if n_c else 0.0
        filas.append({'clase': CLASES[c], 'n_prueba': n_c, 'M_c': M_c, 'm_c': m_c,
                      'flujos_esperado': (m_c / (1 - m_c)) if m_c < 1 else np.inf})
    return pd.DataFrame(filas)

def resumen_cota(cm):
    t = cota_retardo(cm)
    return {'flujos_ataque_como_normal': int(t['M_c'].sum()), 'clases_con_fallos': int((t['M_c'] > 0).sum()),
            'flujos_peor_caso_max': int(t['M_c'].max()), 'flujos_esperado_media': float(t['flujos_esperado'].replace(np.inf, np.nan).mean())}

def tfp_de(y, pred):
    n = y == NORMAL
    return float((pred[n] != NORMAL).mean() * 100)

def pred_tau(pN, att, tau, argmax_pred=None):
    """S5: «Normal» si P(Normal) ≥ τ; si no, la clase de ataque más probable. τ = 'argmax': predicción por defecto."""
    if tau == 'argmax':
        return argmax_pred
    return np.where(pN >= tau, NORMAL, att).astype(np.int64)

def partes_proba(P):
    P = np.asarray(P, dtype=np.float32)
    pN = P[:, NORMAL].copy()
    Pa = P.copy(); Pa[:, NORMAL] = -1
    return pN, Pa.argmax(1).astype(np.int8), P.argmax(1).astype(np.int8)

def elegir_tau(pN, att, amax, y):
    """Regla S5 (fijada): entre los τ (incluido «argmax») con TFP fuera de pliegue ≤ S5_TFP_MAX, el de mayor F1 macro;
    si ninguno cumple, el de menor TFP (empate: mayor F1)."""
    filas = []
    for tau in ['argmax'] + list(S5_TAU_GRID):
        p = pred_tau(pN, att, tau, amax).astype(np.int64)
        filas.append({'tau': tau, 'tfp_oof': tfp_de(y, p), 'f1_oof': f1_macro(y, p)})
    t = pd.DataFrame(filas)
    t['cumple'] = t['tfp_oof'] <= S5_TFP_MAX
    el = (t[t.cumple].sort_values('f1_oof', ascending=False, kind='mergesort').iloc[0] if t.cumple.any()
          else t.sort_values(['tfp_oof', 'f1_oof'], ascending=[True, False], kind='mergesort').iloc[0])
    return t, {'tau': el['tau'], 'tfp_oof': float(el['tfp_oof']), 'f1_oof': float(el['f1_oof']), 'cumple_tfp_max': bool(el['cumple'])}

def cv_T(D, cols, fabricas):
    """CV de 3 pliegues (como v1) sobre S2_T_GRID para cada modelo de `fabricas` ({nombre: callable}).
    Devuelve la tabla y las predicciones fuera de pliegue (pN, mejor ataque, argmax) por (modelo, T)."""
    n = len(D.ytr)
    pliegues = list(StratifiedKFold(3, shuffle=True, random_state=D.semilla).split(np.zeros(n), D.ytr))
    acc, oof = {}, {}
    for f, (a_, b_) in enumerate(pliegues):
        Xv, yv = D.Xtr[np.ix_(b_, cols)], D.ytr[b_]
        for T in S2_T_GRID:
            t0 = time.perf_counter()
            Xb, yb = datos_T(D.Xtr[np.ix_(a_, cols)], D.ytr[a_], T, D.semilla + f)   # semilla + pliegue, como v1
            for nombre, fab in fabricas.items():
                m = fab().fit(Xb, yb)
                pN, att, amax = partes_proba(m.predict_proba(Xv))
                o = oof.setdefault((nombre, T or 0), [np.zeros(n, np.float32), np.zeros(n, np.int8), np.zeros(n, np.int8)])
                o[0][b_], o[1][b_], o[2][b_] = pN, att, amax
                v = acc.setdefault((nombre, T or 0), {'f1': [], 'kb': [], 'nodos': [], 'n_ent': []})
                v['f1'].append(f1_macro(yv, amax.astype(np.int64))); v['kb'].append(kb_joblib(m))
                v['nodos'].append(sum(e.tree_.node_count for e in m.estimators_) if hasattr(m, 'estimators_') else np.nan)
                v['n_ent'].append(len(yb))
                del m
            del Xb, yb
            print(f'   pliegue {f + 1}/3 · T = {T_txt(T)}: {time.perf_counter() - t0:.0f} s')
        del Xv
    tabla = pd.DataFrame([{'modelo': m_, 'T': T, 'f1_macro_cv': np.mean(v['f1']), 'f1_de': np.std(v['f1'], ddof=1),
                           'tamano_kb': np.mean(v['kb']), 'nodos': np.mean(v['nodos']), 'flujos_entrenamiento': np.mean(v['n_ent'])}
                          for (m_, T), v in acc.items()])
    return tabla, oof

def regla_T(tabla):
    """Regla de v1 aplicada a T (por modelo): candidatos con F1 CV ≥ 99 % del mejor; el de menor tamaño joblib (empate: mayor F1)."""
    tabla = tabla.copy(); elec = {}
    tabla['cumple_99'] = False; tabla['elegido'] = False
    for m_, g in tabla.groupby('modelo'):
        mejor = g['f1_macro_cv'].max()
        ok = g['f1_macro_cv'] >= S2_TOLERANCIA * mejor
        tabla.loc[g.index, 'cumple_99'] = ok
        el = g[ok].sort_values(['tamano_kb', 'f1_macro_cv'], ascending=[True, False], kind='mergesort').iloc[0]
        tabla.loc[el.name, 'elegido'] = True
        elec[m_] = {'T': int(el['T']), 'f1_macro_cv': float(el['f1_macro_cv']), 'mejor_f1_cv': float(mejor),
                    'tamano_kb_cv': float(el['tamano_kb'])}
    return tabla, elec

def nuevo_rf(rf_cfg, semilla):
    return RandomForestClassifier(rf_cfg[0], max_depth=rf_cfg[1], n_jobs=-1, random_state=semilla)

def nuevo_xgb(V, semilla):
    return XGBClassifier(n_estimators=V.xgb[0], max_depth=V.xgb[1], learning_rate=V.xgb_lr, tree_method='hist', device='cpu',
                         n_jobs=-1, random_state=semilla, eval_metric='mlogloss')
print('Utilidades de S11d listas.')

## 3. Script de medición (un proceso por modelo, CPU, un hilo)
Generaliza el de S10 a cuatro formatos (`keras`, `joblib`, `onnx`, `tflite`).
- **Lote de 1 024** (como v1): calentamiento con una pasada completa descartada; latencia = tiempo total / N.
- **Lote de 1** (M1): mediana por flujo sobre `n1` flujos tras 500 llamadas de calentamiento. Con `--respuesta` mide además el **tiempo de respuesta por flujo** = selección de los k atributos + min-max + inferencia (para M5-a).
- **Memoria** (RSS con psutil, cada 5 ms): `memoria_pico_mb` = definición de v1 (aumento durante la inferencia con el modelo ya cargado); `memoria_total_mb` = runtime + modelo + inferencia (huella completa del despliegue).

In [ ]:
RUTA_MEDIR = os.path.join(DIR_TMP, 'medir_v2.py')

In [ ]:
%%writefile {RUTA_MEDIR}
import argparse, json, os, threading, time
import numpy as np, psutil

p = argparse.ArgumentParser()
p.add_argument('--formato', choices=['keras', 'joblib', 'onnx', 'tflite'], required=True)
p.add_argument('--tipo', choices=['tcn', 'rf', 'xgb'], required=True)
p.add_argument('--modelo', required=True)
p.add_argument('--x', required=True)
p.add_argument('--salida', required=True)
p.add_argument('--lote', type=int, default=1024)
p.add_argument('--x1', default='')
p.add_argument('--n1', type=int, default=0)
p.add_argument('--calentamiento1', type=int, default=500)
p.add_argument('--respuesta', action='store_true')
a = p.parse_args()
proc = psutil.Process()
rss = lambda: proc.memory_info().rss
MB = 1024 ** 2

X = np.load(a.x)
rss_datos = rss()
t0 = time.perf_counter()
if a.formato == 'keras':
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    import keras
elif a.formato == 'joblib':
    import joblib
    if a.tipo == 'xgb':
        import xgboost
    else:
        import sklearn.ensemble
elif a.formato == 'onnx':
    import onnxruntime as ort
else:
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
rss_runtime = rss()

if a.formato == 'keras':
    modelo = keras.models.load_model(a.modelo)
    def predecir(L):
        return np.argmax(modelo.predict_on_batch(L.reshape(len(L), 1, -1)), axis=1)
elif a.formato == 'joblib':
    modelo = joblib.load(a.modelo)
    if a.tipo == 'rf':
        modelo.set_params(n_jobs=1)
    else:
        modelo.set_params(n_jobs=1, device='cpu')
    def predecir(L):
        return np.asarray(modelo.predict(L)).astype(np.int64).ravel()
elif a.formato == 'onnx':
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    sesion = ort.InferenceSession(a.modelo, so, providers=['CPUExecutionProvider'])
    ent = sesion.get_inputs()[0]
    es_3d = len(ent.shape) == 3
    if a.tipo == 'tcn':
        def predecir(L):
            return np.argmax(sesion.run(None, {ent.name: L.reshape(len(L), 1, -1) if es_3d else L})[0], axis=1)
    else:
        etiqueta = sesion.get_outputs()[0].name
        def predecir(L):
            return np.asarray(sesion.run([etiqueta], {ent.name: L})[0]).astype(np.int64).ravel()
else:
    interp = Interpreter(model_path=a.modelo, num_threads=1)
    e_in, e_out = interp.get_input_details()[0], interp.get_output_details()[0]
    forma = [None]
    def predecir(L):
        L = np.ascontiguousarray(L.reshape(len(L), 1, -1), dtype=np.float32)
        if forma[0] != L.shape:
            interp.resize_tensor_input(e_in['index'], L.shape)
            interp.allocate_tensors()
            forma[0] = L.shape
        interp.set_tensor(e_in['index'], L)
        interp.invoke()
        return np.argmax(interp.get_tensor(e_out['index']), axis=1)
t_carga = time.perf_counter() - t0
rss_modelo = rss()

# --- Lote de 1 024 (como v1) ---------------------------------------------------------------------------
N, B = len(X), a.lote
inicios = list(range(0, N, B))
pred = np.empty(N, dtype=np.int64)

class Muestreo(threading.Thread):
    def __init__(s):
        super().__init__(daemon=True); s.pico = rss(); s.activo = True
    def run(s):
        while s.activo:
            r = rss()
            if r > s.pico:
                s.pico = r
            time.sleep(0.005)

mu = Muestreo(); mu.start()
for i in inicios:                       # calentamiento: una pasada completa descartada
    predecir(X[i:i + B])
c0 = time.perf_counter()
for i in inicios:
    pred[i:i + B] = predecir(X[i:i + B])
total = time.perf_counter() - c0
mu.activo = False; mu.join()

r = {'formato': a.formato, 'tipo': a.tipo, 'n_flujos': N, 'lote': B, 'tiempo_total_s': total,
     'latencia_ms_por_flujo': total / N * 1000, 'tiempo_carga_s': t_carga,
     'rss_datos_mb': rss_datos / MB, 'rss_runtime_mb': rss_runtime / MB, 'rss_modelo_mb': rss_modelo / MB,
     'rss_pico_mb': mu.pico / MB,
     'memoria_pico_mb': (mu.pico - rss_modelo) / MB,
     'memoria_modelo_mb': (rss_modelo - rss_runtime) / MB,
     'memoria_runtime_mb': (rss_runtime - rss_datos) / MB,
     'memoria_total_mb': (mu.pico - rss_datos) / MB,
     'tamano_archivo_kb': os.path.getsize(a.modelo) / 1024}

# --- Lote de 1 (M1) ------------------------------------------------------------------------------------
if a.n1 > 0 and a.x1:
    d = np.load(a.x1)
    Xs = np.ascontiguousarray(d['x_escalado'][:a.n1], dtype=np.float32)
    n1 = len(Xs)
    for i in range(min(a.calentamiento1, n1)):
        predecir(Xs[i:i + 1])
    t1 = np.empty(n1); p1 = np.empty(n1, dtype=np.int64)
    for i in range(n1):
        c = time.perf_counter_ns()
        p1[i] = predecir(Xs[i:i + 1])[0]
        t1[i] = time.perf_counter_ns() - c
    t1 /= 1e6
    r.update(lote1_n=n1, lote1_mediana_ms=float(np.median(t1)), lote1_p95_ms=float(np.percentile(t1, 95)),
             lote1_p99_ms=float(np.percentile(t1, 99)), lote1_media_ms=float(t1.mean()),
             lote1_acuerdo_con_lote1024=float((p1 == pred[d['indices'][:n1]]).mean()))
    if a.respuesta:
        Xc, cols = d['x_crudo'][:n1], d['cols']
        mn, sc = d['min_'], d['scale_']
        t2 = np.empty(n1); p2 = np.empty(n1, dtype=np.int64)
        for i in range(min(a.calentamiento1, n1)):
            z = Xc[i, cols][None, :].astype(np.float32); z *= sc; z += mn
            predecir(z)
        for i in range(n1):
            c = time.perf_counter_ns()
            z = Xc[i, cols][None, :].astype(np.float32)   # selección de los k atributos
            z *= sc; z += mn                               # min-max (igual que MinMaxScaler.transform)
            p2[i] = predecir(z)[0]
            t2[i] = time.perf_counter_ns() - c
        t2 /= 1e6
        r.update(respuesta_mediana_ms=float(np.median(t2)), respuesta_p95_ms=float(np.percentile(t2, 95)),
                 respuesta_acuerdo_con_lote1=float((p2 == p1).mean()))

np.save(a.salida.replace('.json', '_pred.npy'), pred.astype(np.int8))
json.dump(r, open(a.salida, 'w'), indent=2)
print(json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()}))

In [ ]:
ENV_1HILO = dict(os.environ, OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1',
                 TF_NUM_INTRAOP_THREADS='1', TF_NUM_INTEROP_THREADS='1', TF_CPP_MIN_LOG_LEVEL='2')

def medir(formato, tipo, modelo, x, salida, n1=0, x1='', respuesta=False):
    """Mide en un proceso aparte. Reutiliza la medición si ya está guardada (REANUDAR)."""
    if not hay(salida):
        cmd = [sys.executable, RUTA_MEDIR, '--formato', formato, '--tipo', tipo, '--modelo', modelo, '--x', x,
               '--salida', salida, '--lote', str(LOTE_MEDICION)]
        if n1:
            cmd += ['--x1', x1, '--n1', str(n1), '--calentamiento1', str(M1_CALENTAMIENTO_LOTE1)]
        if respuesta:
            cmd += ['--respuesta']
        env = {**ENV_1HILO, 'CUDA_VISIBLE_DEVICES': '-1'} if formato in ('keras', 'tflite') else ENV_1HILO
        p = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if p.returncode != 0:
            print(p.stdout[-3000:], p.stderr[-3000:])
            raise RuntimeError(f'Falló la medición {tipo}/{formato} ({modelo})')
    m = leer_json(salida)
    m['pred'] = np.load(salida.replace('.json', '_pred.npy')).astype(np.int64)
    return m

## Funciones reutilizadas de M1 y M4 (sin cambios)
Solo definen funciones (conversión a ONNX y preprocesamiento sin artefacto); no ejecutan nada.

In [ ]:
DIR_M1 = os.path.join(DIR_MEJ, 'M1_despliegue')

def a_onnx_arbol(modelo, tipo, k, destino):
    if tipo == 'rf':
        from skl2onnx import convert_sklearn
        from skl2onnx.common.data_types import FloatTensorType
        on = convert_sklearn(modelo, initial_types=[('x', FloatTensorType([None, k]))],
                             options={id(modelo): {'zipmap': False}}, target_opset={'': M1_OPSET, 'ai.onnx.ml': 3})
    else:
        import onnxmltools
        from onnxmltools.convert.common.data_types import FloatTensorType
        on = onnxmltools.convert_xgboost(modelo, initial_types=[('x', FloatTensorType([None, k]))], target_opset=15)
    with open(destino, 'wb') as f:
        f.write(on.SerializeToString())
    return destino

_TF = {}
def a_onnx_tcn(ruta_keras, destino, n_atr):
    """Plan A: tf2onnx → keras.export(format='onnx'). Plan B: TFLite. Devuelve (ruta, método)."""
    for ruta, metodo in ((destino, None), (destino.replace('.onnx', '.tflite'), 'tflite')):
        if hay(ruta) and hay(ruta + '.metodo'):
            return ruta, open(ruta + '.metodo').read().strip()
    if not _TF:
        import logging
        import tensorflow as tf
        import keras
        logging.getLogger('tf2onnx').setLevel(logging.ERROR)
        _TF.update(tf=tf, keras=keras)
    tf, keras = _TF['tf'], _TF['keras']
    modelo = keras.models.load_model(ruta_keras)
    errores = []
    try:
        import tf2onnx
        tf2onnx.convert.from_keras(modelo, input_signature=(tf.TensorSpec((None, 1, n_atr), tf.float32, name='x'),),
                                   opset=M1_OPSET, output_path=destino)
        metodo = 'tf2onnx'
    except Exception as e:
        errores.append(f'tf2onnx: {type(e).__name__}: {str(e)[:200]}')
        try:
            modelo.export(destino, format='onnx', verbose=False)
            metodo = 'keras.export'
        except Exception as e2:
            errores.append(f'keras.export: {type(e2).__name__}: {str(e2)[:200]}')
            print('⚠️ La TCN no se pudo exportar a ONNX; se usa el plan B (TFLite).', errores)
            destino = destino.replace('.onnx', '.tflite')
            with open(destino, 'wb') as f:
                f.write(tf.lite.TFLiteConverter.from_keras_model(modelo).convert())
            metodo = 'tflite'
    with open(destino + '.metodo', 'w') as f:
        f.write(metodo)
    return destino, metodo

RUNTIME = {'keras': 'Nativo v1 (Keras)', 'joblib': 'Nativo v1 (scikit-learn / XGBoost)', 'onnx': 'ONNX Runtime', 'tflite': 'TFLite'}

def m1_rep(r):
    dir_r = os.path.join(DIR_M1, f'REP-{r:02d}')
    final = os.path.join(dir_r, 'M1_REP.json')
    if hay(final):
        return leer_json(final)
    t_ini = time.time()
    os.makedirs(dir_r, exist_ok=True)
    D, V = dividir(r), v1(r)
    verificar_v1(D, V)
    tmp = os.path.join(DIR_TMP, 'M1', D.codigo); os.makedirs(tmp, exist_ok=True)
    sub = np.sort(np.random.default_rng(D.semilla).choice(len(D.yte), min(M1_N_LOTE1, len(D.yte)), replace=False))
    XM = {'tcn': guardar_x(os.path.join(tmp, 'x_tcn.npy'), D.Xte[:, IDX_GAO]),
          'k': guardar_x(os.path.join(tmp, 'x_k.npy'), D.Xte[:, V.cols_k])}
    X1 = {'tcn': lote1_npz(os.path.join(tmp, 'x1_tcn.npz'), D, sub, IDX_GAO),
          'k': lote1_npz(os.path.join(tmp, 'x1_k.npz'), D, sub, V.cols_k)}

    # Conversiones (se guardan en Drive: son pequeñas)
    ruta_tcn_conv, metodo_tcn = a_onnx_tcn(V.ruta_tcn, os.path.join(dir_r, 'tcn.onnx'), len(IDX_GAO))
    fmt_tcn = 'tflite' if metodo_tcn == 'tflite' else 'onnx'
    rf, xg = joblib.load(V.ruta_rf), joblib.load(V.ruta_xgb)
    ruta_rf_c3 = os.path.join(dir_r, 'rf_c3.joblib')
    if not hay(ruta_rf_c3):
        joblib.dump(rf, ruta_rf_c3, compress=3)
    ruta_rf_onnx, ruta_xgb_onnx = os.path.join(dir_r, 'rf.onnx'), os.path.join(dir_r, 'xgb.onnx')
    if not hay(ruta_rf_onnx):
        a_onnx_arbol(rf, 'rf', V.K, ruta_rf_onnx)
    if not hay(ruta_xgb_onnx):
        a_onnx_arbol(xg, 'xgb', V.K, ruta_xgb_onnx)
    del rf, xg

    # Mediciones intercaladas en la misma sesión: nativo (v1) y runtime común, por modelo
    plan = [('tcn', 'keras', V.ruta_tcn, 'tcn', M1_N_LOTE1_NATIVO, False),
            ('tcn', fmt_tcn, ruta_tcn_conv, 'tcn', M1_N_LOTE1, True),
            ('rf', 'joblib', V.ruta_rf, 'k', M1_N_LOTE1_NATIVO, False),
            ('rf', 'onnx', ruta_rf_onnx, 'k', M1_N_LOTE1, True),
            ('xgb', 'joblib', V.ruta_xgb, 'k', M1_N_LOTE1_NATIVO, False),
            ('xgb', 'onnx', ruta_xgb_onnx, 'k', M1_N_LOTE1, True)]
    filas, pred_nativo = [], {}
    for tipo, fmt, ruta_m, xs, n1, resp in plan:
        m = medir(fmt, tipo, ruta_m, XM[xs], os.path.join(dir_r, f'medicion_{tipo}_{fmt}.json'), n1=n1, x1=X1[xs], respuesta=resp)
        if fmt in ('keras', 'joblib'):
            pred_nativo[tipo] = m['pred']
        f1 = f1_macro(D.yte, m['pred'])
        fila = {k: v for k, v in m.items() if k != 'pred'}
        fila.update(rep=r, modelo=tipo, runtime=RUNTIME[fmt], f1_macro=f1, f1_v1=V.ind[tipo]['f1_macro'],
                    dif_f1_v1=f1 - V.ind[tipo]['f1_macro'], acuerdo_con_nativo=float((m['pred'] == pred_nativo[tipo]).mean()),
                    tamano_gzip_kb=kb_gzip(ruta_m), metodo_conversion=(metodo_tcn if tipo == 'tcn' and fmt != 'keras' else
                                                                    ('skl2onnx' if tipo == 'rf' and fmt == 'onnx' else
                                                                     ('onnxmltools' if fmt == 'onnx' else '—'))))
        if tipo == 'rf' and fmt == 'joblib':
            fila['tamano_joblib_c3_kb'] = os.path.getsize(ruta_rf_c3) / 1024
        filas.append(fila)
    res = {'rep': r, 'semilla': D.semilla, 'metodo_tcn': metodo_tcn, 'filas': filas, 'entorno': ENTORNO,
           'segundos': round(time.time() - t_ini, 1)}
    guardar_json(res, final)
    bitacora('M1', r, t_ini)
    shutil.rmtree(tmp, ignore_errors=True)
    del D; gc.collect()
    return res

In [ ]:
DIR_M4 = os.path.join(DIR_MEJ, 'M4_sin_artefacto')
_BASE_U = {}

def base_unificada():
    if not _BASE_U:
        B = preprocesar(unificar_ceros=True)
        assert np.array_equal(B.y, Y), 'Las filas cambiaron al unificar: no debería pasar'
        _BASE_U['B'] = B
    return _BASE_U['B']

def m4_rep(r):
    dir_r = os.path.join(DIR_M4, 'corridas', f'REP-{r:02d}')
    ruta_ficha = os.path.join(dir_r, f'ficha_REP-{r:02d}.json')
    if hay(ruta_ficha):
        return leer_json(ruta_ficha)
    t_ini = time.time(); os.makedirs(dir_r, exist_ok=True)
    BU = base_unificada()
    D, V = dividir(r), v1(r)
    verificar_v1(D, V)
    DU = dividir(r, BU)
    assert np.array_equal(DU.idx_te, D.idx_te), 'La división no coincide con v1'
    tmp = os.path.join(DIR_TMP, 'M4', D.codigo); os.makedirs(tmp, exist_ok=True)
    # Configuración
    if M4_CONFIG == 'v2':
        e = leer_json(os.path.join(DIR_M3, 'corridas', f'REP-{r:02d}', 'seleccion_m3.json'))
        K, T, RF = e['k'], M3_T, (e['arboles'], e['profundidad'])
        su = M3_FILTRO_SU
    else:
        K, T, RF, su = V.K, V.T, V.rf, False
    t_cv = time.perf_counter()
    rank, orden = ranking_im(DU, BU.ATRIBUTOS, su=su)
    t_mi = time.perf_counter() - t_cv
    rank.to_csv(os.path.join(dir_r, 'mi_ranking.csv'), index=False)
    cols = orden[:K]
    t0 = time.perf_counter()
    Xb, yb = balancear(DU.Xtr[:, cols], DU.ytr, T, D.semilla)
    t_bal = time.perf_counter() - t0
    rf = RandomForestClassifier(RF[0], max_depth=RF[1], n_jobs=-1, random_state=D.semilla)
    xg = XGBClassifier(n_estimators=V.xgb[0], max_depth=V.xgb[1], learning_rate=V.xgb_lr, tree_method='hist', device='cpu',
                       n_jobs=-1, random_state=D.semilla, eval_metric='mlogloss')
    tiempos_ent = {}
    for nombre, modelo in [('rf', rf), ('xgb', xg)]:
        t0 = time.perf_counter(); modelo.fit(Xb, yb); tiempos_ent[nombre] = time.perf_counter() - t0
    ruta_rf, ruta_xgb = os.path.join(dir_r, 'rf.joblib'), os.path.join(dir_r, 'xgb.joblib')
    joblib.dump(rf, ruta_rf); joblib.dump(xg, ruta_xgb)
    finales = {'flujos_entrenamiento_balanceado': int(len(yb)), 'tiempo_balanceo_s': round(t_bal, 1), 'xgb_device': 'cpu'}
    del Xb, yb, rf, xg
    res = medir_ficha(D, V, tmp, dir_r, ruta_rf, DU.Xte[:, cols], ruta_xgb, DU.Xte[:, cols])
    sel = {'k': int(K), 'T': int(T), 'rf_arboles': int(RF[0]), 'rf_profundidad': int(RF[1]), 'xgb_arboles': V.xgb[0],
           'xgb_profundidad': V.xgb[1], 'xgb_lr': V.xgb_lr, 'atributos_k': [BU.ATRIBUTOS[j] for j in cols],
           'tiempo_cv_s': round(t_mi, 1), 'config': M4_CONFIG, 'filtro_su': su}
    tiempos = {'entrenamiento_tcn_min': V.ficha['tiempos']['entrenamiento_tcn_min'], 'validacion_cruzada_min': t_mi / 60,
               'entrenamiento_rf_s': tiempos_ent['rf'], 'entrenamiento_xgb_s': tiempos_ent['xgb']}
    ficha = construir_ficha(D, V, res, {'tcn': len(IDX_GAO), 'rf': int(K), 'xgb': int(K)}, sel, finales, tiempos,
                            f'M4: sin artefacto «0»/«0.0» (config {M4_CONFIG})', BU.preproc, len(BU.ATRIBUTOS))
    ficha['control'] = {'tcn_f1_igual_v1': abs(res['tcn']['f1_macro'] - V.ind['tcn']['f1_macro']) < 0.01}
    guardar_json(ficha, ruta_ficha)
    bitacora('M4', r, t_ini)
    shutil.rmtree(tmp, ignore_errors=True)
    del D, DU; gc.collect()
    return ficha

## 3.1 Utilidades de S11e

In [ ]:
# S11e: rutas y utilidades
DIR_E = os.path.join(DIR_MEJ, 'S11e_paper')
os.makedirs(DIR_E, exist_ok=True)
TODAS_REPS = list(range(1, 11))

def ficha_de(dir_parte, r):
    return leer_json(os.path.join(dir_parte, 'corridas', f'REP-{r:02d}', f'ficha_REP-{r:02d}.json'))

def ataque_normal_pct(cm):
    cm = np.asarray(cm)
    return float(cm[ATAQUES, NORMAL].sum() / cm[ATAQUES].sum() * 100)

def t_corregida(d, n_tr, n_te):
    """t remuestreada corregida (Nadeau y Bengio, 2003): var × (1/J + n_prueba/n_entrenamiento)."""
    d = np.asarray(d, float); J = len(d); m = float(d.mean()); s2 = float(d.var(ddof=1))
    se = np.sqrt((1 / J + n_te / n_tr) * s2)
    t = m / se if se > 0 else np.inf
    h = stats.t.ppf(0.975, J - 1) * se
    return {'media_d': m, 't_corr': float(t), 'gl': J - 1, 'p_corr': float(2 * stats.t.sf(abs(t), J - 1)),
            'ic95_corr': (round(float(m - h), 4), round(float(m + h), 4))}

def lote1_npz_u(ruta, DU, sub, cols, Xcrudo):
    cols = np.asarray(cols)
    np.savez(ruta, x_escalado=DU.Xte[sub][:, cols], x_crudo=Xcrudo[DU.idx_te[sub]], cols=cols, indices=sub,
             min_=DU.esc.min_[cols], scale_=DU.esc.scale_[cols])
    return ruta

def idx_gao_unificado(BU):
    return [BU.ATRIBUTOS.index(S3_MAPEO_GAO.get(c, c)) for c in GAO_18]

def ms(x):
    x = np.asarray(x, float)
    return [float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0]
print('Salida de S11e:', DIR_E)

In [ ]:
def hay_gpu():
    return bool(shutil.which('nvidia-smi')) and 'GPU' in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout

## E8. LSTM y Transformer compactos (GPU)
Mismas particiones (semilla 1000 + r), datos sin artefacto y las **25 entradas de IM** de MI-LiteXGB en cada REP (las de su ficha de S3).
Cada modelo se entrena en un proceso aparte que también predice la prueba; los de la REP-01 se exportan a ONNX para E9.

In [ ]:
DIR_E8 = os.path.join(DIR_MEJ, 'S14_recurrentes')
os.makedirs(DIR_E8, exist_ok=True)
RUTA_E8 = os.path.join(DIR_TMP, 'entrenar_e8.py')
SCRIPT_E8 = r'''
import argparse, json, time
import numpy as np
p = argparse.ArgumentParser()
for a_ in ('--x', '--y', '--xte', '--pred', '--info', '--cfg', '--arq', '--onnx'):
    p.add_argument(a_, default='')
p.add_argument('--variante')
p.add_argument('--semilla', type=int)
p.add_argument('--n_cls', type=int)
a = p.parse_args()
import tensorflow as tf
import keras
from keras import layers, ops
cfg, arq = json.loads(a.cfg), json.loads(a.arq)


def construir_lstm(n_atr, n_cls, unidades):
    """LSTM compacta: cada atributo es un paso de una secuencia de longitud n_atr con un canal."""
    ent = keras.Input(shape=(n_atr,), name='x')
    x = layers.Reshape((n_atr, 1))(ent)
    x = layers.LSTM(unidades)(x)
    sal = layers.Dense(n_cls, activation='softmax')(x)
    return keras.Model(ent, sal, name='LSTM_compacta')


class TokensAtributo(layers.Layer):
    """Tokenizador de atributos (estilo FT-Transformer): cada atributo j tiene su propio peso y sesgo de dimensión d,
    y se antepone un token [CLS] aprendido."""
    def __init__(self, n_atr, d, **kw):
        super().__init__(**kw)
        self.n_atr, self.d = n_atr, d

    def build(self, forma):
        self.w = self.add_weight(shape=(self.n_atr, self.d), initializer='glorot_uniform', name='w')
        self.b = self.add_weight(shape=(self.n_atr, self.d), initializer='zeros', name='b')
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer='glorot_uniform', name='cls')

    def call(self, x):
        t = ops.expand_dims(x, -1) * self.w + self.b                       # (lote, n_atr, d)
        cls = ops.broadcast_to(self.cls, (ops.shape(x)[0], 1, self.d))      # (lote, 1, d)
        return ops.concatenate([cls, t], axis=1)

    def get_config(self):
        return {**super().get_config(), 'n_atr': self.n_atr, 'd': self.d}


def construir_transformer(n_atr, n_cls, d, cabezas, capas, ffn, dropout):
    """Transformer compacto (pre-norma) sobre los tokens de atributo; clasifica con el token [CLS]."""
    ent = keras.Input(shape=(n_atr,), name='x')
    x = TokensAtributo(n_atr, d)(ent)
    for _ in range(capas):
        h = layers.LayerNormalization(epsilon=1e-6)(x)
        h = layers.MultiHeadAttention(num_heads=cabezas, key_dim=d // cabezas, dropout=dropout)(h, h)
        x = layers.Add()([x, h])
        h = layers.LayerNormalization(epsilon=1e-6)(x)
        h = layers.Dense(ffn, activation='relu')(h)
        h = layers.Dropout(dropout)(h)
        h = layers.Dense(d)(h)
        x = layers.Add()([x, h])
    x = layers.LayerNormalization(epsilon=1e-6)(x[:, 0, :])
    sal = layers.Dense(n_cls, activation='softmax')(x)
    return keras.Model(ent, sal, name='Transformer_compacto')


keras.utils.set_random_seed(a.semilla)
X, y = np.load(a.x), np.load(a.y)
n_atr = X.shape[1]
if a.variante == 'lstm':
    modelo = construir_lstm(n_atr, a.n_cls, **arq)
elif a.variante == 'transformer':
    modelo = construir_transformer(n_atr, a.n_cls, **arq)
else:
    raise ValueError(a.variante)
modelo.compile(optimizer=keras.optimizers.Adam(cfg['lr']), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg['paciencia'], restore_best_weights=True)] if cfg['paciencia'] else []
t0 = time.perf_counter()
hist = modelo.fit(X, y, epochs=cfg['epocas'], batch_size=cfg['lote'], validation_split=cfg['val_split'], shuffle=True,
                  verbose=2, callbacks=cb)
t_ent = time.perf_counter() - t0
Xte = np.load(a.xte)
pred = np.argmax(modelo.predict(Xte, batch_size=8192, verbose=0), axis=1).astype(np.int8)
np.save(a.pred, pred)
info = {'tiempo_entrenamiento_s': round(t_ent, 1), 'parametros': int(modelo.count_params()), 'config': cfg, 'arquitectura': arq,
        'epocas_corridas': len(hist.history['loss']), 'gpu': [g.name for g in tf.config.list_physical_devices('GPU')],
        'historia': {k: [float(v) for v in vs] for k, vs in hist.history.items()}}
if a.onnx:   # REP-01: exportar a ONNX para medir la latencia con el mismo runtime que los árboles y la TCN
    try:
        import tf2onnx
        modelo.output_names = ['salida']
        tf2onnx.convert.from_keras(modelo, input_signature=(tf.TensorSpec((None, n_atr), tf.float32, name='x'),), opset=17,
                                   output_path=a.onnx)
        info['onnx'] = 'tf2onnx'
    except Exception as e:
        try:
            modelo.export(a.onnx, format='onnx', verbose=False)
            info['onnx'] = 'keras.export'
        except Exception as e2:
            info['onnx'] = f'falló: {type(e).__name__}: {str(e)[:150]} | {type(e2).__name__}: {str(e2)[:150]}'
    try:
        import onnxruntime as ort
        s = ort.InferenceSession(a.onnx, providers=['CPUExecutionProvider'])
        po = np.argmax(s.run(None, {s.get_inputs()[0].name: Xte[:5000].astype(np.float32)})[0], axis=1)
        info['onnx_acuerdo'] = float((po == pred[:5000]).mean())
    except Exception as e:
        info['onnx_acuerdo'] = f'no verificado: {type(e).__name__}'
json.dump(info, open(a.info, 'w'), indent=2)
print(a.variante, 'listo:', round(t_ent / 60, 1), 'min ·', info['parametros'], 'parámetros ·', info.get('onnx', ''))
'''
with open(RUTA_E8, 'w') as f:
    f.write(SCRIPT_E8)

def e8_rep(r, variante):
    dir_v = os.path.join(DIR_E8, variante, f'REP-{r:02d}')
    ruta = os.path.join(dir_v, 'resultado.json')
    if hay(ruta):
        return leer_json(ruta)
    t_ini = time.time(); os.makedirs(dir_v, exist_ok=True)
    BU = base_unificada(); DU = dividir(r, BU)
    cols = [BU.ATRIBUTOS.index(a) for a in ficha_de(DIR_S3, r)['seleccion']['atributos_k']]
    tmp = os.path.join(DIR_TMP, 'E8', variante, f'REP-{r:02d}'); os.makedirs(tmp, exist_ok=True)
    x = guardar_x(os.path.join(tmp, 'xtr.npy'), DU.Xtr[:, cols]); y = os.path.join(tmp, 'ytr.npy'); np.save(y, DU.ytr)
    xte = guardar_x(os.path.join(tmp, 'xte.npy'), DU.Xte[:, cols]); pred = os.path.join(tmp, 'pred.npy')
    info = os.path.join(dir_v, 'entrenamiento.json'); log = os.path.join(tmp, 'log.txt')
    onnx = os.path.join(dir_v, f'{variante}.onnx') if r == 1 else ''
    arq = E8_LSTM if variante == 'lstm' else E8_TRANSFORMER
    cmd = [sys.executable, RUTA_E8, '--x', x, '--y', y, '--xte', xte, '--pred', pred, '--info', info, '--cfg', json.dumps(E8_CFG),
           '--arq', json.dumps(arq), '--onnx', onnx, '--variante', variante, '--semilla', str(DU.semilla), '--n_cls', str(N_CLASES)]
    env = dict(os.environ, TF_FORCE_GPU_ALLOW_GROWTH='true', TF_CPP_MIN_LOG_LEVEL='2')
    print(f'   {variante}: entrenando (avance en {log})')
    p = subprocess.run(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=env)
    if p.returncode != 0 or not os.path.exists(pred):
        print('\n'.join(open(log).read().splitlines()[-40:]))
        raise RuntimeError(f'Falló {variante} en la REP-{r:02d}')
    ind = indicadores(DU.yte, np.load(pred).astype(np.int64))
    ent = leer_json(info)
    res = {'rep': r, 'variante': variante, 'k': len(cols), **ind, 'ataque_normal_pct': ataque_normal_pct(ind['matriz_confusion']),
           'entrenamiento': {k: v for k, v in ent.items() if k != 'historia'}}
    guardar_json(res, ruta); shutil.copy(log, dir_v)
    bitacora('E8', r, t_ini, variante)
    shutil.rmtree(tmp, ignore_errors=True)
    return res

if correr('E8'):
    pendientes = [(r, v) for v in E8_VARIANTES for r in REPS if not hay(os.path.join(DIR_E8, v, f'REP-{r:02d}', 'resultado.json'))]
    if pendientes and not hay_gpu():
        raise SystemExit('⛔ E8 necesita GPU (Entorno de ejecución → Cambiar tipo de entorno → L4 o A100, RAM alta).')
    E8R = {}
    for r in REPS:
        for v in E8_VARIANTES:
            t0 = time.time(); E8R[(v, r)] = d = e8_rep(r, v)
            print(f"REP-{r:02d} {v}: F1 {d['f1_macro']:.2f} · TFP {d['tfp']:.3f} % · ataques como Normal {d['ataque_normal_pct']:.2f} % · "
                  f"épocas {d['entrenamiento']['epocas_corridas']} · {d['entrenamiento']['parametros']:,} parámetros · {time.time() - t0:.0f} s")


## E9. Latencia por paquete en la misma sesión (REP-01, ONNX Runtime, 1 hilo)
LSTM y Transformer (E8) junto con la TCN, MI-LiteRF y MI-LiteXGB del protocolo C (archivos ONNX de S3): lote de 1 (mediana sobre
10 000 paquetes de prueba tras 500 llamadas de calentamiento) y tiempo por paquete con lotes de 1 024. Cada modelo en su propio proceso.

In [ ]:
RUTA_BENCH_E9 = os.path.join(DIR_TMP, 'bench_e9.py')
with open(RUTA_BENCH_E9, 'w') as f:
    f.write(r'''
import sys, json, time, numpy as np, onnxruntime as ort, resource
modelo, xpath, salida, cal = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4])
so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
s = ort.InferenceSession(modelo, so, providers=['CPUExecutionProvider'])
ent = s.get_inputs()[0]; es3 = len(ent.shape) == 3
X = np.load(xpath).astype(np.float32)
def pr(x):
    return s.run(None, {ent.name: x.reshape(len(x), 1, -1) if es3 else x})[0]
for i in range(min(cal, len(X))):
    pr(X[i:i + 1])
t = np.empty(len(X))
for i in range(len(X)):
    c = time.perf_counter_ns(); pr(X[i:i + 1]); t[i] = time.perf_counter_ns() - c
B = 1024; pr(X[:B]); tb = []
for i in range(0, len(X) - B + 1, B):
    c = time.perf_counter_ns(); pr(X[i:i + B]); tb.append((time.perf_counter_ns() - c) / B)
json.dump({'lote1_mediana_us': float(np.median(t) / 1e3), 'lote1_p99_us': float(np.percentile(t, 99) / 1e3),
           'lote1024_us_por_paquete': float(np.median(tb) / 1e3), 'pico_mb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024},
          open(salida, 'w'))
''')

RUTA_E9 = os.path.join(DIR_E8, 'E9_latencia.json')
if correr('E9'):
    if hay(RUTA_E9):
        E9 = leer_json(RUTA_E9)
    else:
        BU = base_unificada(); DU = dividir(1, BU)
        cols = [BU.ATRIBUTOS.index(a) for a in ficha_de(DIR_S3, 1)['seleccion']['atributos_k']]
        sub = np.sort(np.random.default_rng(DU.semilla).choice(len(DU.yte), min(E9_N_LOTE1, len(DU.yte)), replace=False))
        x_mi = guardar_x(os.path.join(DIR_TMP, 'e9_x_mi.npy'), DU.Xte[sub][:, cols])
        x_gao = guardar_x(os.path.join(DIR_TMP, 'e9_x_gao.npy'), DU.Xte[sub][:, idx_gao_unificado(BU)])
        dir3 = os.path.join(DIR_S3, 'corridas', 'REP-01')
        plan = [('LSTM', os.path.join(DIR_E8, 'lstm', 'REP-01', 'lstm.onnx'), x_mi),
                ('Transformer', os.path.join(DIR_E8, 'transformer', 'REP-01', 'transformer.onnx'), x_mi),
                ('TCN', os.path.join(dir3, 'tcn_u.onnx'), x_gao),
                ('MI-LiteRF', os.path.join(dir3, 'rf_u3.onnx'), x_mi),
                ('MI-LiteXGB', os.path.join(dir3, 'xgb_u3.onnx'), x_mi)]
        E9 = {'procesador': cpu_modelo(), 'onnxruntime': version('onnxruntime'), 'n': int(len(sub)), 'modelos': {}}
        for nombre, ruta_m, xp in plan:
            if not os.path.exists(ruta_m):
                print(f'   {nombre}: falta {ruta_m}, se omite'); continue
            sal = os.path.join(DIR_TMP, f'e9_{nombre}.json')
            p = subprocess.run([sys.executable, RUTA_BENCH_E9, ruta_m, xp, sal, str(E9_CALENTAMIENTO)], env=ENV_1HILO,
                               capture_output=True, text=True)
            if p.returncode != 0:
                print(f'   {nombre}: falló la medición', p.stderr[-500:]); continue
            E9['modelos'][nombre] = {**leer_json(sal), 'onnx_kb': os.path.getsize(ruta_m) / 1024}
            m = E9['modelos'][nombre]
            print(f"   {nombre:11s} lote 1: {m['lote1_mediana_us']:.1f} µs (p99 {m['lote1_p99_us']:.1f}) · "
                  f"lote 1 024: {m['lote1024_us_por_paquete']:.2f} µs/paquete · ONNX {m['onnx_kb']:.0f} KB")
        guardar_json(E9, RUTA_E9)


## E10. Resumen y pruebas pareadas
Media ± DE por modelo en las REP disponibles, junto con la TCN de Gao (sus 18 entradas), la TCN con las 25 entradas de IM (E5 de S11e)
y MI-LiteXGB. Pruebas con la t remuestreada corregida (Holm sobre las cuatro comparaciones de F1 macro).
**Comparte `S14_recurrentes_resultados.json`** para integrarlo al artículo.

In [ ]:
if correr('E10'):
    N_TR, N_TE = 1_336_769, 572_902
    def leer_opc(ruta):
        return leer_json(ruta) if os.path.exists(ruta) else None
    R = {v: {r: leer_opc(os.path.join(DIR_E8, v, f'REP-{r:02d}', 'resultado.json')) for r in range(1, 11)} for v in E8_VARIANTES}
    TMI = {r: leer_opc(os.path.join(DIR_E, 'E5_tcn', 'tcn_mi25', f'REP-{r:02d}', 'resultado.json')) for r in range(1, 11)}
    reps = [r for r in range(1, 11) if all(R[v][r] for v in E8_VARIANTES)]
    F3 = {r: ficha_de(DIR_S3, r) for r in reps}
    print('REP completas:', reps)
    def fila(nombre, d, r):
        return {'modelo': nombre, 'rep': r, 'f1_macro': d['f1_macro'], 'tfp': d['tfp'], 'ataque_normal_pct': d['ataque_normal_pct']}
    filas = []
    for r in reps:
        for v, nombre in (('lstm', 'LSTM (MI top-25)'), ('transformer', 'Transformer (MI top-25)')):
            filas.append({**fila(nombre, R[v][r], r), 'epocas': R[v][r]['entrenamiento']['epocas_corridas'],
                          'parametros': R[v][r]['entrenamiento']['parametros'],
                          'tiempo_min': R[v][r]['entrenamiento']['tiempo_entrenamiento_s'] / 60})
        if TMI[r]:
            filas.append(fila('TCN (MI top-25)', TMI[r], r))
        for m, nombre in (('tcn_u', "TCN (Gao's 18)"), ('xgb_u3', 'MI-LiteXGB')):
            d = F3[r]['modelos'][m]
            filas.append({'modelo': nombre, 'rep': r, 'f1_macro': d['f1_macro'], 'tfp': d['tfp'],
                          'ataque_normal_pct': ataque_normal_pct(F3[r]['matrices_confusion'][m])})
    T10 = pd.DataFrame(filas)
    resumen = {m: {k: ms(g[k].dropna()) for k in ('f1_macro', 'tfp', 'ataque_normal_pct', 'epocas', 'parametros', 'tiempo_min') if k in g and g[k].notna().any()}
               for m, g in T10.groupby('modelo', sort=False)}
    for m in resumen:
        resumen[m]['n_rep'] = int((T10['modelo'] == m).sum())
    g = lambda nombre: T10[T10['modelo'] == nombre].set_index('rep')['f1_macro']
    pruebas = []
    for a_, b_ in (('LSTM (MI top-25)', 'MI-LiteXGB'), ('Transformer (MI top-25)', 'MI-LiteXGB'),
                   ('TCN (MI top-25)', 'LSTM (MI top-25)'), ('TCN (MI top-25)', 'Transformer (MI top-25)')):
        if len(g(a_)) >= 3 and len(g(b_)) >= 3:
            comunes = sorted(set(g(a_).index) & set(g(b_).index))
            d = (g(b_).loc[comunes] - g(a_).loc[comunes]).to_numpy()
            pruebas.append({'comparacion': f'{a_} → {b_}', 'reps': comunes, **t_corregida(d, N_TR, N_TE)})
    for x, ph in zip(pruebas, holm([x['p_corr'] for x in pruebas])):
        x['p_holm'] = float(ph)
    pc = {}
    for v in E8_VARIANTES:
        if reps:
            pc[v] = {'f1_por_clase': {c: float(np.mean([R[v][r]['f1_por_clase'][c] for r in reps])) for c in CLASES},
                     'matriz_sumada': np.sum([np.array(R[v][r]['matriz_confusion']) for r in reps], axis=0).tolist()}
    S = {'generado': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'), 'entorno': ENTORNO, 'reps': reps,
         'configuracion': {'cfg': E8_CFG, 'lstm': E8_LSTM, 'transformer': E8_TRANSFORMER},
         'resumen': resumen, 'pruebas_f1': pruebas, 'por_clase': pc,
         'latencia': leer_opc(os.path.join(DIR_E8, 'E9_latencia.json')), 'por_rep': T10.to_dict(orient='records')}
    ruta = os.path.join(DIR_E8, 'S14_recurrentes_resultados.json'); guardar_json(S, ruta)
    print()
    for m, v in resumen.items():
        extra = f" · {v['parametros'][0]:,.0f} parámetros · {v['epocas'][0]:.0f} épocas" if 'parametros' in v else ''
        print(f"{m:24s} F1 macro {v['f1_macro'][0]:.2f} ± {v['f1_macro'][1]:.2f} · TFP {v['tfp'][0]:.3f} ± {v['tfp'][1]:.3f} % · "
              f"ataques como Normal {v['ataque_normal_pct'][0]:.2f} %{extra} · n = {v['n_rep']}")
    for x in pruebas:
        print(f"  {x['comparacion']}: Δ F1 {x['media_d']:+.2f} {x['ic95_corr']} · p_corr (Holm) {x['p_holm']:.2g}")
    print('\n✔ Guardado:', ruta)
